In [1]:
# bootstrap_ci_map50_yolo_v6.py
#
# Purpose: Compute 95% bootstrap confidence interval for test set mAP50
#
# Why bootstrap CI:
#   Your test set mAP50=0.532 is a single point estimate from 59 tiles.
#   With only 59 tiles there is uncertainty around this number.
#   Bootstrap estimates this uncertainty by asking:
#   "If I had a slightly different set of 59 test tiles, what mAP50
#   would I get?" It does this 1000 times by resampling tiles WITH
#   replacement and recomputing mAP50 each time.
#   The 2.5th and 97.5th percentile of those 1000 values = 95% CI.
#
# How it works step by step:
#   1. Load all 59 prediction files and 59 ground truth files
#   2. Repeat 1000 times:
#      a. Sample 59 tile indices WITH replacement
#         (some tiles appear twice, some not at all)
#      b. Collect all predictions and ground truths for those tiles
#      c. Sort all predictions by confidence (highest first)
#      d. Walk through predictions: TP if IoU>=0.5 with unmatched GT
#      e. Build PR curve, compute area = AP50 per class
#      f. Average AP50 across classes = mAP50 for this bootstrap sample
#   3. Report 2.5th and 97.5th percentile of 1000 mAP50 values

import os
import numpy as np
import pandas as pd

# ── paths ──────────────────────────────────────────────────────────────────────
# prediction files — saved by model.val(save_txt=True, save_conf=True)
# each file has 300 lines: class cx cy w h confidence (all normalized 0-1)
SUMMER_PRED_DIR = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation_yolo_v6\test_summer\labels"
FALL_PRED_DIR   = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\test_set_evaluation_yolo_v6\test_fall\labels"

# ground truth files — original YOLO format labels for 59 test tiles
# each line: class cx cy w h (no confidence — these are the real annotations)
SUMMER_GT_DIR   = r"F:\Masterthesis\dataset\cv_folds_2cls\test\labels"
FALL_GT_DIR     = r"F:\Masterthesis\dataset\cv_folds_fall_2cls\test\labels"

OUTPUT_DIR = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\statistical_analysis_yolo_v6"
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_BOOTSTRAP = 1000  # number of resampling iterations
IOU_THRESH  = 0.5   # IoU threshold for counting a detection as correct
CLASSES     = [0, 1]  # 0=deciduous_tree, 1=shrub
SEED        = 42

# ── helper functions ───────────────────────────────────────────────────────────

def compute_iou(pred, gt):
    """
    Compute IoU between predicted and ground truth box.
    Both in format [cx, cy, w, h] normalized 0-1.
    Converts to corners, computes intersection over union.
    """
    def corners(b):
        return b[0]-b[2]/2, b[1]-b[3]/2, b[0]+b[2]/2, b[1]+b[3]/2

    px1,py1,px2,py2 = corners(pred)
    gx1,gy1,gx2,gy2 = corners(gt)

    ix1,iy1 = max(px1,gx1), max(py1,gy1)
    ix2,iy2 = min(px2,gx2), min(py2,gy2)

    if ix2<=ix1 or iy2<=iy1:
        return 0.0

    inter = (ix2-ix1)*(iy2-iy1)
    union = (px2-px1)*(py2-py1) + (gx2-gx1)*(gy2-gy1) - inter
    return inter/union if union > 0 else 0.0


def compute_ap50(preds, gts_dict, n_gt):
    """
    Compute AP50 for one class from a list of predictions.

    preds:    list of (confidence, cx, cy, w, h, tile_idx) sorted by conf desc
    gts_dict: dict of tile_idx → list of [cx,cy,w,h] ground truth boxes
    n_gt:     total number of ground truth boxes for this class

    Returns AP50 (area under PR curve at IoU>=0.5)
    """
    if n_gt == 0 or not preds:
        return 0.0

    # track which gt boxes already matched
    matched = {idx: [False]*len(boxes) for idx, boxes in gts_dict.items()}

    tp = []
    fp = []

    for conf, cx, cy, w, h, tile_idx in preds:
        gt_boxes = gts_dict.get(tile_idx, [])

        best_iou = 0.0
        best_j   = -1
        for j, gt in enumerate(gt_boxes):
            iou = compute_iou([cx,cy,w,h], gt)
            if iou > best_iou:
                best_iou = iou
                best_j   = j

        if best_iou >= IOU_THRESH and best_j >= 0 and not matched[tile_idx][best_j]:
            tp.append(1)
            fp.append(0)
            matched[tile_idx][best_j] = True
        else:
            tp.append(0)
            fp.append(1)

    tp_cs = np.cumsum(tp)
    fp_cs = np.cumsum(fp)
    rec   = np.concatenate([[0], tp_cs / n_gt,          [1]])
    prec  = np.concatenate([[1], tp_cs / (tp_cs+fp_cs+1e-8), [0]])

    return float(np.trapezoid(prec, rec))


def load_data(pred_dir, gt_dir):
    """
    Load predictions and ground truths for all tiles.
    Returns per-class data structures ready for bootstrap resampling.
    """
    gt_files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".txt")])
    n_tiles  = len(gt_files)
    print(f"  Found {n_tiles} ground truth files")

    # per-class storage
    preds_by_class = {c: {} for c in CLASSES}  # tile_idx → list of (conf,cx,cy,w,h)
    gts_by_class   = {c: {} for c in CLASSES}  # tile_idx → list of [cx,cy,w,h]

    for tile_idx, fname in enumerate(gt_files):
        # ── load ground truth ──────────────────────────────────────────────────
        for c in CLASSES:
            gts_by_class[c][tile_idx] = []

        gt_path = os.path.join(gt_dir, fname)
        with open(gt_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                if c in CLASSES:
                    gts_by_class[c][tile_idx].append(
                        [float(parts[1]), float(parts[2]),
                         float(parts[3]), float(parts[4])]
                    )

        # ── load predictions ───────────────────────────────────────────────────
        for c in CLASSES:
            preds_by_class[c][tile_idx] = []

        pred_path = os.path.join(pred_dir, fname)
        if not os.path.exists(pred_path):
            continue

        with open(pred_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6:
                    continue
                c    = int(parts[0])
                conf = float(parts[5])
                if c in CLASSES:
                    preds_by_class[c][tile_idx].append(
                        (conf, float(parts[1]), float(parts[2]),
                               float(parts[3]), float(parts[4]))
                    )

    return preds_by_class, gts_by_class, n_tiles


def bootstrap_map50(preds_by_class, gts_by_class, n_tiles, label):
    """
    Run bootstrap resampling to estimate 95% CI for mAP50.
    """
    np.random.seed(SEED)
    map50_samples = []

    print(f"\n  Running {N_BOOTSTRAP} bootstrap iterations...")

    for i in range(N_BOOTSTRAP):
        if i % 200 == 0:
            print(f"    Iteration {i}/{N_BOOTSTRAP}...")

        # sample tile indices WITH replacement
        sampled = np.random.choice(n_tiles, size=n_tiles, replace=True)

        ap_scores = []
        for c in CLASSES:
            # build bootstrap predictions and ground truths
            boot_preds  = []
            boot_gts    = {}
            n_gt        = 0

            for new_idx, orig_idx in enumerate(sampled):
                boot_gts[new_idx] = gts_by_class[c].get(orig_idx, [])
                n_gt += len(boot_gts[new_idx])

                for conf, cx, cy, w, h in preds_by_class[c].get(orig_idx, []):
                    boot_preds.append((conf, cx, cy, w, h, new_idx))

            # sort by confidence descending — critical for correct PR curve
            boot_preds.sort(key=lambda x: x[0], reverse=True)

            ap = compute_ap50(boot_preds, boot_gts, n_gt)
            ap_scores.append(ap)

        map50_samples.append(np.mean(ap_scores))

    samples   = np.array(map50_samples)
    ci_lower  = np.percentile(samples, 2.5)
    ci_upper  = np.percentile(samples, 97.5)
    mean_boot = np.mean(samples)

    print(f"\n  {label} bootstrap results:")
    print(f"    Bootstrap mean mAP50:  {mean_boot:.4f}")
    print(f"    95% CI:                [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"    CI width:              {ci_upper-ci_lower:.4f}")

    return mean_boot, ci_lower, ci_upper, samples


# ── run bootstrap for summer ───────────────────────────────────────────────────
print("="*60)
print("BOOTSTRAP CI — SUMMER TEST SET (reported mAP50=0.5322)")
print("="*60)
print("Loading data...")
preds_s, gts_s, n_s = load_data(SUMMER_PRED_DIR, SUMMER_GT_DIR)
mean_s, lo_s, hi_s, samples_s = bootstrap_map50(preds_s, gts_s, n_s, "Summer")

# ── run bootstrap for fall ─────────────────────────────────────────────────────
print("\n" + "="*60)
print("BOOTSTRAP CI — FALL TEST SET (reported mAP50=0.2052)")
print("="*60)
print("Loading data...")
preds_f, gts_f, n_f = load_data(FALL_PRED_DIR, FALL_GT_DIR)
mean_f, lo_f, hi_f, samples_f = bootstrap_map50(preds_f, gts_f, n_f, "Fall")

# ── final summary ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\nSummer test set:")
print(f"  Reported mAP50:  0.5322")
print(f"  Bootstrap mean:  {mean_s:.4f}")
print(f"  95% CI:          [{lo_s:.4f}, {hi_s:.4f}]")
print(f"\nFall test set:")
print(f"  Reported mAP50:  0.2052")
print(f"  Bootstrap mean:  {mean_f:.4f}")
print(f"  95% CI:          [{lo_f:.4f}, {hi_f:.4f}]")
print(f"\nNote: Bootstrap CI quantifies uncertainty due to the")
print(f"limited test set size of 59 tiles.")

# ── save ───────────────────────────────────────────────────────────────────────
pd.DataFrame({
    "season":          ["summer", "fall"],
    "reported_mAP50":  [0.5322,   0.2052],
    "bootstrap_mean":  [round(mean_s,4), round(mean_f,4)],
    "ci_lower_95":     [round(lo_s,4),   round(lo_f,4)],
    "ci_upper_95":     [round(hi_s,4),   round(hi_f,4)],
    "ci_width":        [round(hi_s-lo_s,4), round(hi_f-lo_f,4)],
    "n_bootstrap":     [N_BOOTSTRAP, N_BOOTSTRAP],
    "n_test_tiles":    [59, 59],
}).to_csv(os.path.join(OUTPUT_DIR, "bootstrap_ci_results.csv"), index=False)

print(f"\nSaved: {OUTPUT_DIR}/bootstrap_ci_results.csv")
print("\nNote: F: drive must be connected for GT files.")

BOOTSTRAP CI — SUMMER TEST SET (reported mAP50=0.5322)
Loading data...
  Found 59 ground truth files

  Running 1000 bootstrap iterations...
    Iteration 0/1000...
    Iteration 200/1000...
    Iteration 400/1000...
    Iteration 600/1000...
    Iteration 800/1000...

  Summer bootstrap results:
    Bootstrap mean mAP50:  0.5313
    95% CI:                [0.4594, 0.6095]
    CI width:              0.1502

BOOTSTRAP CI — FALL TEST SET (reported mAP50=0.2052)
Loading data...
  Found 59 ground truth files

  Running 1000 bootstrap iterations...
    Iteration 0/1000...
    Iteration 200/1000...
    Iteration 400/1000...
    Iteration 600/1000...
    Iteration 800/1000...

  Fall bootstrap results:
    Bootstrap mean mAP50:  0.2315
    95% CI:                [0.1757, 0.2977]
    CI width:              0.1220

FINAL SUMMARY

Summer test set:
  Reported mAP50:  0.5322
  Bootstrap mean:  0.5313
  95% CI:          [0.4594, 0.6095]

Fall test set:
  Reported mAP50:  0.2052
  Bootstrap mean:  0